In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
df = pd.read_csv('/kaggle/input/datasets/cokunkaanaki/loanapproval-risk/loanapproval.csv')
df['dti'] = df['loan_amount'] / df['annual_income']

# ============================================================
# 1. ALT-SKORLARI HESAPLA (her biri 0-100, 100 = düşük risk)
# ============================================================

# Kredi skoru: min-max normalize
df['score_credit'] = (df['credit_score'] - df['credit_score'].min()) / \
                      (df['credit_score'].max() - df['credit_score'].min()) * 100

# İstihdam durumu: EDA'daki gerçek onay oranlarına göre puanlama
# (Employed %82, Self-employed %85, Unemployed %51 onay oranı gözlemlenmişti)
emp_map = {'Employed': 80, 'Self-employed': 90, 'Unemployed': 20}
df['score_employment'] = df['employment_status'].map(emp_map)

# DTI (borç/gelir): düşük DTI = yüksek puan. %100'ü aşanları sınırla.
dti_capped = df['dti'].clip(upper=1.0)
df['score_dti'] = (1 - dti_capped) * 100

# Mevcut kredi sayısı ve bağımlı sayısı: ters orantılı normalize
df['score_existing_loans'] = (1 - df['existing_loans_count'] / df['existing_loans_count'].max()) * 100
df['score_dependents'] = (1 - df['num_dependents'] / df['num_dependents'].max()) * 100

print("### Alt-skor özetleri ###")
print(df[['score_credit','score_employment','score_dti',
          'score_existing_loans','score_dependents']].describe().round(1))

# ============================================================
# 2. AĞIRLIKLANDIRMA VE TOPLAM RİSK SKORU
# ============================================================

weights = {
    'score_credit':         0.35,
    'score_employment':     0.20,
    'score_dti':            0.20,
    'score_existing_loans': 0.15,
    'score_dependents':     0.10,
}
assert abs(sum(weights.values()) - 1.0) < 1e-9, "Ağırlıklar toplamı 1 olmalı"

df['risk_score'] = sum(df[col] * w for col, w in weights.items())

df['risk_tier'] = pd.cut(
    df['risk_score'], bins=[0, 50, 75, 100],
    labels=['Yüksek Risk', 'Orta Risk', 'Düşük Risk']
)

print("\n### İlk 10 başvurunun risk skoru ###")
print(df[['applicant_id','credit_score','employment_status','dti',
           'risk_score','risk_tier']].head(10).to_string(index=False))

# ============================================================
# 3. RİSK SKORU DAĞILIMI (grafik)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

sns.histplot(df['risk_score'], bins=30, kde=True, ax=axes[0], color='#4C72B0')
axes[0].set_title('Risk Skoru Dağılımı (0-100)')
axes[0].set_xlabel('Risk Skoru (yüksek = düşük risk)')

df['risk_tier'].value_counts().reindex(['Yüksek Risk','Orta Risk','Düşük Risk']).plot(
    kind='bar', ax=axes[1], color=['#C44E52','#DD8452','#55A868'])
axes[1].set_title('Risk Tier Dağılımı')
axes[1].set_xlabel('')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('risk_dagilimi.png', dpi=120)
plt.show()

# ============================================================
# 4. VALİDASYON: skor gerçek onay kararlarını yansıtıyor mu?
# =========================================================

validation = df.groupby('risk_tier', observed=True)['loan_approved'].agg(
    Onay_Orani='mean', Basvuru_Sayisi='count'
).round(3)
print("\n### Validasyon: Risk Tier vs Gerçek Onay Oranı ###")
print(validation)

plt.figure(figsize=(6,4))
validation['Onay_Orani'].reindex(['Yüksek Risk','Orta Risk','Düşük Risk']).plot(
    kind='bar', color=['#C44E52','#DD8452','#55A868'])
plt.title('Risk Tier Bazında Gerçekleşen Onay Oranı')
plt.ylabel('Onay Oranı')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('validasyon.png', dpi=120)
plt.show()

# ============================================================
# 5. EN RİSKLİ / EN GÜVENLİ 10 BAŞVURU
# ============================================================

cols = ['applicant_id','credit_score','employment_status','dti',
        'existing_loans_count','num_dependents','risk_score','loan_approved']

print("\n### En riskli 10 başvuru ###")
print(df.sort_values('risk_score').head(10)[cols].to_string(index=False))

print("\n### En güvenli 10 başvuru ###")
print(df.sort_values('risk_score', ascending=False).head(10)[cols].to_string(index=False))